# Audit causal_data — bối cảnh product–store–week

Tệp lớn được đọc từng chunk. Mục tiêu là kiểm tra mã và khả năng nối với transaction trong **phạm vi train** (DAY ≤ 537), không tạo feature hoặc khẳng định household đã tiếp xúc quảng cáo. Chỉ kiểm tra duplicate cho các khóa trùng transaction train; không kết luận tính duy nhất của mọi khóa trong toàn bộ causal_data.

In [1]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
if ROOT.name == 'research' and ROOT.parent.name == 'notebooks': ROOT = ROOT.parent
if ROOT.name=='notebooks':ROOT=ROOT.parent
OUT=ROOT/'artifacts'/'research'/'causal_context_audit'
cols=['PRODUCT_ID','STORE_ID','WEEK_NO']
tx=pd.read_csv(ROOT/'sample_data'/'dunnhumby'/'transaction_data.csv',usecols=cols+['DAY'])
tx=tx.loc[tx.DAY<=537,cols].copy()
keys=tx.drop_duplicates()
assert not keys.duplicated(cols).any()
print('Transaction train:',len(tx),'| Khóa duy nhất:',len(keys))

Transaction train: 1888242 | Khóa duy nhất: 1722750


## 1. Mã display/mailer và khóa khớp

Mã `0`, giá trị rỗng và missing được ghi nhận riêng; không suy missing là không có trưng bày. Tập causal_data có thể chỉ là các dòng có bối cảnh được ghi nhận, vì vậy không tự coi transaction không khớp là không khuyến mại. Kết nối theo đúng ba cột với khóa transaction được loại trùng trước khi merge để tránh nhân dòng.

In [2]:
counts={'display':Counter(),'mailer':Counter()}; matched_counts={'display':Counter(),'mailer':Counter()}
missing_keys=0;total_rows=0;parts=[]
path=ROOT/'sample_data'/'dunnhumby'/'causal_data.csv'
for chunk in pd.read_csv(path,usecols=cols+['display','mailer'],chunksize=1_000_000,
                         dtype={'PRODUCT_ID':'int64','STORE_ID':'int32','WEEK_NO':'int16','display':'string','mailer':'string'},keep_default_na=False):
    total_rows+=len(chunk)
    missing_keys+=int(chunk[cols].isna().any(axis=1).sum())
    for c in ['display','mailer']:
        counts[c].update(chunk[c].fillna('<NA>').value_counts(dropna=False).to_dict())
    matched=chunk.merge(keys,on=cols,how='inner',validate='many_to_one')
    for c in ['display','mailer']:
        matched_counts[c].update(matched[c].fillna('<NA>').value_counts(dropna=False).to_dict())
    if len(matched):parts.append(matched.groupby(cols,as_index=False).size())
    print('Đã đọc',f'{total_rows:,}','dòng; dòng khớp chunk:',len(matched))
matched_key_counts=(pd.concat(parts,ignore_index=True).groupby(cols,as_index=False)['size'].sum()
                    if parts else pd.DataFrame(columns=cols+['size']))
assert not matched_key_counts.duplicated(cols).any()
key_coverage=keys.merge(matched_key_counts,on=cols,how='left',validate='one_to_one')
transaction_coverage=tx.merge(matched_key_counts,on=cols,how='left',validate='many_to_one')
audit=pd.Series({'causal_rows':total_rows,'causal_missing_key_rows':missing_keys,'unique_transaction_keys':len(keys),
                 'matched_transaction_keys':int(key_coverage['size'].notna().sum()),
                 'matched_transaction_key_rate':key_coverage['size'].notna().mean(),
                 'matched_transaction_rows':int(transaction_coverage['size'].notna().sum()),
                 'matched_transaction_row_rate':transaction_coverage['size'].notna().mean(),
                 'matched_keys_with_multiple_causal_rows':int(key_coverage['size'].fillna(0).gt(1).sum()),
                 'transaction_rows_with_multiple_causal_matches':int(transaction_coverage['size'].fillna(0).gt(1).sum())})
code_counts=pd.DataFrame([{'field':field,'code':repr(code),'all_causal_rows':count,
                           'matched_causal_rows':matched_counts[field].get(code,0)}
                          for field,counter in counts.items() for code,count in counter.items()]).sort_values(['field','all_causal_rows'],ascending=[True,False])
display(audit.to_frame('kết quả'));display(code_counts)

Đã đọc 1,000,000 dòng; dòng khớp chunk: 0
Đã đọc 2,000,000 dòng; dòng khớp chunk: 0
Đã đọc 3,000,000 dòng; dòng khớp chunk: 0
Đã đọc 4,000,000 dòng; dòng khớp chunk: 0
Đã đọc 5,000,000 dòng; dòng khớp chunk: 0
Đã đọc 6,000,000 dòng; dòng khớp chunk: 22034
Đã đọc 7,000,000 dòng; dòng khớp chunk: 29626
Đã đọc 8,000,000 dòng; dòng khớp chunk: 23747
Đã đọc 9,000,000 dòng; dòng khớp chunk: 26103
Đã đọc 10,000,000 dòng; dòng khớp chunk: 21108
Đã đọc 11,000,000 dòng; dòng khớp chunk: 20661
Đã đọc 12,000,000 dòng; dòng khớp chunk: 28366
Đã đọc 13,000,000 dòng; dòng khớp chunk: 21397
Đã đọc 14,000,000 dòng; dòng khớp chunk: 22689
Đã đọc 15,000,000 dòng; dòng khớp chunk: 19910
Đã đọc 16,000,000 dòng; dòng khớp chunk: 23442
Đã đọc 17,000,000 dòng; dòng khớp chunk: 18947
Đã đọc 18,000,000 dòng; dòng khớp chunk: 0
Đã đọc 19,000,000 dòng; dòng khớp chunk: 0
Đã đọc 20,000,000 dòng; dòng khớp chunk: 0
Đã đọc 21,000,000 dòng; dòng khớp chunk: 0
Đã đọc 22,000,000 dòng; dòng khớp chunk: 0
Đã đọc 23,000,0

,kết quả
causal_rows,3.678652e+07
causal_missing_key_rows,0.000000e+00
unique_transaction_keys,1.722750e+06
matched_transaction_keys,3.576510e+05
matched_transaction_key_rate,2.076047e-01
matched_transaction_rows,4.180160e+05
matched_transaction_row_rate,2.213784e-01
matched_keys_with_multiple_causal_rows,4.260000e+02
transaction_rows_with_multiple_causal_matches,5.870000e+02


,field,code,all_causal_rows,matched_causal_rows
0,display,'0',21038745,180505
2,display,'9',2699467,24918
3,display,'5',2575289,27436
1,display,'7',2362118,33886
4,display,'3',2073738,25077
7,display,'6',1816021,11485
5,display,'2',1812840,27189
8,display,'1',1102141,12878
6,display,'A',713180,10026
9,display,'4',592985,4677


**Kết quả:** Trong transaction DAY ≤ 537, 20,76% khóa product–store–week và 22,14% dòng transaction có ít nhất một dòng causal khớp. Có 426 khóa transaction khớp nhiều dòng causal, ảnh hưởng 587 dòng transaction nếu nối trực tiếp; chưa xác định các bản ghi đó trùng hệt hay chứa mã khác nhau. Đây không phải tỷ lệ household tiếp xúc hoặc tỷ lệ khuyến mại; không khớp chưa được xác định nghĩa.


## 1.1. Bản ghi khớp nhiều dòng có cùng mã hay khác mã?

Trích lại các dòng causal cho các khóa bị lặp trong phạm vi transaction train; so sánh cặp `(display, mailer)` trên toàn bộ dòng của mỗi khóa. Nếu các cặp giống nhau thì chúng trùng nhau trên **năm cột đã đọc** (ba khóa và hai mã). Nếu cặp khác nhau, không tự chọn một dòng hoặc giả định nguyên nhân.

Theo User Guide trang 6, `display = 0` nghĩa là không được trưng bày; `mailer = 0` nghĩa là không có trong quảng cáo. Trạng thái transaction không khớp vẫn khác mã `0`.

In [3]:
duplicate_keys=matched_key_counts.loc[matched_key_counts['size'].gt(1),cols].copy()
assert len(duplicate_keys)==int(audit['matched_keys_with_multiple_causal_rows'])
dup_parts=[]
for chunk in pd.read_csv(path,usecols=cols+['display','mailer'],chunksize=1_000_000,
                         dtype={'PRODUCT_ID':'int64','STORE_ID':'int32','WEEK_NO':'int16','display':'string','mailer':'string'},keep_default_na=False):
    matched=chunk.merge(duplicate_keys,on=cols,how='inner',validate='many_to_one')
    if len(matched):dup_parts.append(matched)
dup_rows=pd.concat(dup_parts,ignore_index=True) if dup_parts else pd.DataFrame(columns=cols+['display','mailer'])
if len(dup_rows):
    distinct=dup_rows.drop_duplicates(cols+['display','mailer']).groupby(cols,as_index=False).size().rename(columns={'size':'distinct_code_pairs'})
    duplicate_detail=matched_key_counts.merge(distinct,on=cols,how='inner',validate='one_to_one')
    assert len(duplicate_detail)==len(duplicate_keys) and dup_rows.shape[0]==int(duplicate_detail['size'].sum())
    duplicate_detail['classification']=np.where(duplicate_detail.distinct_code_pairs.eq(1),
                                               'Cùng cặp mã trên 5 cột','Cặp mã khác nhau')
    affected=tx.merge(duplicate_detail[cols+['classification']],on=cols,how='inner',validate='many_to_one')
    detail_summary=(duplicate_detail.groupby('classification',as_index=False)
                    .agg(keys=('size','size'),causal_rows=('size','sum')))
    affected_summary=affected.groupby('classification').size().rename('transaction_rows').reset_index()
    detail_summary=detail_summary.merge(affected_summary,on='classification',how='left',validate='one_to_one')
else:
    duplicate_detail=pd.DataFrame(columns=cols+['size','distinct_code_pairs','classification'])
    detail_summary=pd.DataFrame(columns=['classification','keys','causal_rows','transaction_rows'])
display(detail_summary)
print('Khóa lặp đã kiểm tra:',len(duplicate_detail),'| Dòng causal tương ứng:',len(dup_rows))

Khóa lặp đã kiểm tra: 426 | Dòng causal tương ứng: 852


,classification,keys,causal_rows,transaction_rows
0,Cặp mã khác nhau,426,852,587


In [4]:
patterns=dup_rows.groupby(cols,as_index=False).agg(distinct_display=('display','nunique'),distinct_mailer=('mailer','nunique'))
patterns['pattern']=np.select([(patterns.distinct_display>1)&(patterns.distinct_mailer>1), patterns.distinct_display>1],
                              ['Cả display và mailer','Chỉ display'],default='Chỉ mailer')
pattern_summary=patterns.groupby('pattern').size().rename('keys').reset_index()
display(pattern_summary)

,pattern,keys
0,Chỉ display,425
1,Cả display và mailer,1


**Kết quả:** Cả 426 khóa đều có hai cặp mã khác nhau; 425 khóa khác `display` và một khóa khác cả `display` lẫn `mailer`. Không có khóa nào trùng hoàn toàn trên năm cột. Không thể tự động deduplicate bằng cách giữ một dòng.


**Giới hạn xử lý:** Phân loại trên không tự động chứng minh cặp mã nào đúng hoặc được biết trước cutoff. Việc khử trùng lặp chỉ khả thi với khóa có cùng cặp mã sau khi xác nhận không có cột nguồn nào khác cần giữ; không được dùng quy tắc đó cho khóa có cặp mã khác nhau. Chưa thay đổi pipeline.

## 2. Giới hạn và bàn giao

Không có household_key trong causal_data. Một dòng trùng chỉ mô tả bối cảnh của sản phẩm/cửa hàng/tuần, không chứng minh hộ nhìn thấy quảng cáo. Không khớp và mã `0` là hai trạng thái riêng; chưa xác minh tính sẵn có tại cutoff hay độ trễ công bố. Trước khi dùng feature, cần quyết định cách xử lý duplicate, tính đầy đủ, nguồn âm tính và kiểm tra theo thời gian. Kết quả audit không tự chứng minh quan hệ nhân quả.

In [5]:
OUT.mkdir(parents=True,exist_ok=True)
audit.to_csv(OUT/'audit_summary.csv',header=True,encoding='utf-8-sig')
code_counts.to_csv(OUT/'code_counts.csv',index=False,encoding='utf-8-sig')
duplicate_detail.to_csv(OUT/'duplicate_key_details.csv',index=False,encoding='utf-8-sig')
detail_summary.to_csv(OUT/'duplicate_key_summary.csv',index=False,encoding='utf-8-sig')
dup_rows.to_csv(OUT/'duplicate_causal_rows.csv',index=False,encoding='utf-8-sig')
pattern_summary.to_csv(OUT/'duplicate_code_patterns.csv',index=False,encoding='utf-8-sig')
print('Đã lưu:',OUT)

Đã lưu: artifacts\causal_context_audit
